**Tim Pengembang:**

Ahmad Nidzomunnashil - NIM 607012400122

Vikry Achmad Sonjaya - NIM 607012400001

Mardini Dwi Putri - NIM 607012430015

Kelas: 48-02

**Project: Mental Health Risk Prediction**

Notebook ini melatih dan mengevaluasi tiga model klasifikasi (KNN, SVM, Decision Tree) untuk memprediksi tingkat risiko kesehatan mental (0=Rendah, 1=Sedang, 2=Tinggi).

# Load library yang dibutuhkan

In [ ]:
from mpl_toolkits.axisartist.axislines import SubplotZero  # import SubplotZero untuk kebutuhan visualisasi sumbu khusus
import matplotlib.pyplot as plt  # import matplotlib untuk membuat grafik
import numpy as np  # import numpy untuk operasi numerik
import pandas as pd  # import pandas untuk membaca dan mengelola data tabel
import seaborn  # import seaborn untuk mempercantik tampilan grafik
seaborn.set(style='ticks')  # mengatur gaya tampilan grafik seaborn
import matplotlib.cm as cm  # import colormap matplotlib
from sklearn import preprocessing  # import modul preprocessing dari sklearn
from sklearn.preprocessing import MinMaxScaler  # import MinMaxScaler sebagai alternatif normalisasi data
from sklearn.preprocessing import StandardScaler  # import StandardScaler untuk standarisasi fitur
from sklearn.preprocessing import LabelEncoder  # import LabelEncoder untuk mengubah label kategorik menjadi angka
from sklearn.model_selection import train_test_split  # import fungsi untuk membagi data train dan test
from sklearn.model_selection import GridSearchCV  # import GridSearchCV untuk hyperparameter tuning
from sklearn.neighbors import KNeighborsClassifier  # import KNN Classifier
from sklearn.svm import SVC  # import Support Vector Classifier
from sklearn.tree import DecisionTreeClassifier  # import Decision Tree Classifier
from sklearn.metrics import classification_report  # import classification report untuk evaluasi
from sklearn.metrics import accuracy_score  # import fungsi accuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # import fungsi confusion matrix dan visualisasinya

# Load dataset

Dataset diakses dari Google Drive (file id: `1BRSI-0XdgOpB1GUaM3lilUuFrKtRmO0f`). Sel di bawah memakai `gdown` agar dataset langsung ter-download di Colab tanpa perlu mount Drive.

In [ ]:
# Download dataset dari Google Drive
import gdown  # import gdown untuk download dari Google Drive
file_id = '1JvF0t0eg2K7OtcewYpwAXILtjpd3UK78'  # ID file dataset di Google Drive
output_path = 'mental_helth_risk_dataset_1000.csv'  # nama file lokal hasil download
gdown.download(f'https://drive.google.com/uc?id={file_id}', output_path, quiet=False)  # eksekusi download

Downloading...
From: https://drive.google.com/uc?id=1JvF0t0eg2K7OtcewYpwAXILtjpd3UK78
To: /content/mental_helth_risk_dataset_1000.csv
100%|██████████| 851k/851k [00:00<00:00, 11.9MB/s]


'mental_helth_risk_dataset_1000.csv'

In [ ]:
# Load dataset ke dalam DataFrame
df = pd.read_csv(output_path)  # membaca file CSV menjadi DataFrame
print('Shape dataset:', df.shape)  # menampilkan jumlah baris dan kolom
df.head()  # menampilkan 5 baris pertama untuk melihat isi data

Shape dataset: (10000, 25)


,age,gender,marital_status,education_level,employment_status,sleep_hours,physical_activity_hours_per_week,screen_time_hours_per_day,social_support_score,work_stress_level,...,depression_score,stress_level,mood_swings_frequency,concentration_difficulty_level,panic_attack_history,family_history_mental_illness,previous_mental_health_diagnosis,therapy_history,substance_use,mental_health_risk
0,56,Other,Single,Bachelor,Unemployed,8.6,2.8,9.6,7,10,...,4,8,8,3,1,0,1,1,1,1
1,47,Male,Single,Bachelor,Unemployed,4.5,2.7,3.0,10,6,...,7,4,9,3,0,0,0,0,0,0
2,56,Female,Divorced,Bachelor,Student,3.1,14.1,7.2,10,5,...,3,1,4,2,1,1,1,1,1,2
3,59,Other,Married,Bachelor,Employed,7.0,0.5,10.3,2,10,...,8,5,2,5,1,1,0,1,1,2
4,58,Male,Single,High School,Self-Employed,5.1,2.5,1.2,8,1,...,8,3,3,1,0,0,1,0,1,0


In [ ]:
# Cek missing values
print('Total missing values:', df.isnull().sum().sum())  # cek total missing value
df.isnull().sum()[df.isnull().sum() > 0]  # tampilkan kolom yang punya missing value

Total missing values: 0


,0


# Ambil feature dan label dari dataset

Fitur kategorikal di-encode sesuai sifat datanya:
- `education_level` → **ordinal mapping** (High School < Bachelor < Master < PhD) karena ada urutan jenjang
- `gender`, `marital_status`, `employment_status` → **one-hot encoding** karena tidak ada urutan alami

In [ ]:
df_encoded = df.copy()

# Ordinal encoding untuk education_level (terendah → tertinggi)
education_order = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
df_encoded['education_level'] = df_encoded['education_level'].map(education_order)
print(f'education_level (ordinal): {education_order}')

# One-hot encoding untuk kolom kategorik tanpa urutan alami
kolom_ohe = ['gender', 'marital_status', 'employment_status']
df_encoded = pd.get_dummies(df_encoded, columns=kolom_ohe, dtype=int)
ohe_cols = [c for c in df_encoded.columns if any(c.startswith(k + '_') for k in kolom_ohe)]
print(f'\nKolom hasil one-hot encoding: {ohe_cols}')

education_level (ordinal): {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}

Kolom hasil one-hot encoding: ['gender_Female', 'gender_Male', 'gender_Other', 'marital_status_Divorced', 'marital_status_Married', 'marital_status_Single', 'employment_status_Employed', 'employment_status_Self-Employed', 'employment_status_Student', 'employment_status_Unemployed']


In [ ]:
# Ambil fitur (X) dan label (y)
X = df_encoded.drop(columns=['mental_health_risk'])  # X = semua kolom kecuali target
y = df_encoded['mental_health_risk']  # y = kolom target

print('Shape X:', X.shape)  # menampilkan ukuran data fitur
print('Shape y:', y.shape)  # menampilkan ukuran data target
print('Jumlah fitur:', X.shape[1])  # jumlah fitur input

Shape X: (10000, 31)
Shape y: (10000,)
Jumlah fitur: 31


# Membagi dataset untuk data training (70%) dan data testing (30%)

In [ ]:
# Split data menjadi 70 persen training dan 30 persen testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)  # split data dengan stratifikasi agar proporsi kelas tetap
print(X_train.shape)  # menampilkan ukuran data training
print(X_test.shape)  # menampilkan ukuran data testing

(7000, 31)
(3000, 31)


# Feature scaling


In [ ]:
# Standarisasi fitur menggunakan StandardScaler
scaler = StandardScaler()  # membuat objek StandardScaler
scaler.fit(X_train)  # mempelajari rata-rata dan standar deviasi dari data training
X_train = scaler.transform(X_train)  # mengubah data training ke skala standar
X_test = scaler.transform(X_test)  # mengubah data testing dengan skala yang sama (jangan fit ulang)

# Model 1: K-Nearest Neighbors (KNN)

## Membuat dan training classifier KNN

Memakai k=39 (hasil HPO yang sesuai dengan train_models.py).

In [ ]:
# Membuat classifier KNN dengan k=39 (sesuai hasil HPO di train_models.py)
classifier_knn = KNeighborsClassifier(n_neighbors=21, metric='manhattan')  # KNN dengan jumlah tetangga = 39 (hasil HPO)

# Melakukan training
classifier_knn.fit(X_train, y_train)  # melatih KNN dengan data training

KNeighborsClassifier(metric='manhattan', n_neighbors=21)

In [ ]:
# Menghitung akurasi KNN berdasarkan data uji
akurasi_knn = classifier_knn.score(X_test, y_test)  # menghitung akurasi pada data testing
print(f'Tingkat Akurasi KNN: {akurasi_knn * 100:.2f}%')  # menampilkan akurasi dalam persen

Tingkat Akurasi KNN: 63.60%


## HPO untuk KNN

Mencari nilai k terbaik dengan GridSearchCV.

In [ ]:
# Definisikan grid parameter untuk KNN
param_grid_knn = {  # daftar parameter yang dicoba
    'n_neighbors': [99],  # variasi nilai k
    'metric': ['euclidean', 'manhattan'],  # variasi metric jarak
}

# GridSearchCV untuk KNN
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, scoring='recall_macro', cv=5, refit=True, verbose=3)  # search parameter terbaik dengan 5-fold CV
grid_knn.fit(X_train, y_train)  # latih dengan semua kombinasi parameter

print('Parameter terbaik KNN:', grid_knn.best_params_)  # tampilkan parameter terbaik
print('Best CV score      :', round(grid_knn.best_score_, 4))  # tampilkan score CV terbaik

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV 1/5] END ...metric=euclidean, n_neighbors=1;, score=0.480 total time=   0.1s
[CV 2/5] END ...metric=euclidean, n_neighbors=1;, score=0.495 total time=   0.2s
[CV 3/5] END ...metric=euclidean, n_neighbors=1;, score=0.478 total time=   0.1s
[CV 4/5] END ...metric=euclidean, n_neighbors=1;, score=0.466 total time=   0.1s
[CV 5/5] END ...metric=euclidean, n_neighbors=1;, score=0.487 total time=   0.2s
[CV 1/5] END ...metric=euclidean, n_neighbors=3;, score=0.473 total time=   0.3s
[CV 2/5] END ...metric=euclidean, n_neighbors=3;, score=0.493 total time=   0.3s
[CV 3/5] END ...metric=euclidean, n_neighbors=3;, score=0.474 total time=   0.3s
[CV 4/5] END ...metric=euclidean, n_neighbors=3;, score=0.474 total time=   0.2s
[CV 5/5] END ...metric=euclidean, n_neighbors=3;, score=0.479 total time=   0.4s
[CV 1/5] END ...metric=euclidean, n_neighbors=5;, score=0.499 total time=   0.5s
[CV 2/5] END ...metric=euclidean, n_neighbors=

In [ ]:
# Menghitung akurasi KNN berdasarkan data uji
akurasi_grid_knn = accuracy_score(y_test, grid_knn.predict(X_test))  # menghitung akurasi pada data testing
print(f'Tingkat Akurasi KNN: {akurasi_grid_knn * 100:.2f}%')  # menampilkan akurasi dalam persen

Tingkat Akurasi KNN: 65.40%


# Model 2: Support Vector Machine (SVM)

## HPO untuk SVM

Memakai GridSearchCV untuk mencari kombinasi C, gamma, dan kernel terbaik.

In [ ]:
# Definisikan parameter range untuk SVC
param_grid_svm = [  # daftar parameter yang akan diuji
    {'C': [10], 'gamma': [0.01], 'kernel': ['rbf']},  # kombinasi parameter SVC
]

# GridSearchCV untuk SVM
classifier_svm = GridSearchCV(SVC(probability=True), param_grid_svm, scoring='recall_macro', cv=3, refit=True, verbose=3)  # pencarian parameter terbaik
classifier_svm.fit(X_train, y_train)  # melatih model SVC untuk semua kombinasi parameter

print('Parameter terbaik SVM:', classifier_svm.best_params_)  # menampilkan parameter terbaik
print('Best estimator       :', classifier_svm.best_estimator_)  # model terbaik setelah tuning

Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV 1/3] END ......C=10, gamma=0.01, kernel=rbf;, score=0.744 total time=  12.2s
[CV 2/3] END ......C=10, gamma=0.01, kernel=rbf;, score=0.761 total time=   7.4s
[CV 3/3] END ......C=10, gamma=0.01, kernel=rbf;, score=0.771 total time=  10.7s
Parameter terbaik SVM: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Best estimator       : SVC(C=10, gamma=0.01, probability=True)


In [ ]:
# Prediksi data testing dengan SVM
predictions_svm = classifier_svm.predict(X_test)  # memprediksi kelas pada data testing
akurasi_svm = accuracy_score(y_test, predictions_svm)  # menghitung akurasi
print(f'Tingkat Akurasi SVM: {akurasi_svm * 100:.2f}%')  # tampilkan akurasi
print()
print(classification_report(y_test, predictions_svm, target_names=['Rendah', 'Sedang', 'Tinggi']))  # classification report

Tingkat Akurasi SVM: 79.37%

              precision    recall  f1-score   support

      Rendah       0.83      0.84      0.83      1153
      Sedang       0.77      0.79      0.78      1394
      Tinggi       0.77      0.69      0.73       453

    accuracy                           0.79      3000
   macro avg       0.79      0.77      0.78      3000
weighted avg       0.79      0.79      0.79      3000



##SVM non HPO

### SVM Manual Parameter Evaluation

This section manually evaluates different combinations of C, gamma, and kernel for an SVC model, calculating the macro recall for both the training and testing sets. This allows for a detailed inspection of each combination's performance.

In [ ]:

# Membuat classifier SVM tanpa HPO (menggunakan nilai parameter manual/tertentu)
classifier_svm_nh = SVC(kernel='rbf', C=10, gamma=0.01, random_state=0)

# Melakukan training
classifier_svm_nh.fit(X_train, y_train) # melatih SVM dengan data training

SVC(C=100, gamma=0.01, random_state=0)

In [ ]:
predictions_svm_nh = classifier_svm_nh.predict(X_test)  # memprediksi kelas pada data testing
akurasi_svm_nh = accuracy_score(y_test, predictions_svm_nh)  # menghitung akurasi
print(f'Tingkat Akurasi SVM: {akurasi_svm_nh * 100:.2f}%')  # tampilkan akurasi
print()
print(classification_report(y_test, predictions_svm_nh, target_names=['Rendah', 'Sedang', 'Tinggi']))  # classification repo

Tingkat Akurasi SVM: 79.37%

              precision    recall  f1-score   support

      Rendah       0.83      0.84      0.83      1153
      Sedang       0.77      0.79      0.78      1394
      Tinggi       0.77      0.69      0.73       453

    accuracy                           0.79      3000
   macro avg       0.79      0.77      0.78      3000
weighted avg       0.79      0.79      0.79      3000



### Decision Tree Non-HPO: Looping Seluruh Kombinasi (72 Variasi)
Bagian ini mengevaluasi setiap kemungkinan kombinasi parameter secara eksplisit untuk melihat performa model tanpa otomasi HPO formal.

# Model 3: Decision Tree

## HPO untuk Decision Tree

Mencari kombinasi `max_depth`, `min_samples_split`, dan `criterion` terbaik.

In [ ]:
# Definisikan grid parameter untuk Decision Tree
param_grid_dt = {  # daftar parameter yang dicoba
    'criterion': ['gini', 'entropy'],  # kriteria pembagian node
    'max_depth': [5, 10, 15, None],  # kedalaman maksimum tree
    'min_samples_split': [2, 5, 10],  # jumlah minimum sample untuk split node
    'min_samples_leaf': [1, 2, 5],  # jumlah minimum sample dari setiap leaf
}
# GridSearchCV untuk Decision Tree
classifier_dt = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid_dt, scoring='recall_macro', cv=5, refit=True, verbose=1)  # cari parameter terbaik
classifier_dt.fit(X_train, y_train)  # latih dengan semua kombinasi

print('Parameter terbaik DT:', classifier_dt.best_params_)  # parameter terbaik
print('Best estimator      :', classifier_dt.best_estimator_)  # model terbaik

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Parameter terbaik DT: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5}
Best estimator      : DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=2,
                       min_samples_split=5, random_state=0)


In [ ]:
# Prediksi data testing dengan Decision Tree
predictions_dt = classifier_dt.predict(X_test)  # prediksi kelas
akurasi_dt = accuracy_score(y_test, predictions_dt)  # akurasi
print(f'Tingkat Akurasi DT: {akurasi_dt * 100:.2f}%')  # tampilkan akurasi
print()
print(classification_report(y_test, predictions_dt, target_names=['Rendah', 'Sedang', 'Tinggi']))  # classification report

Tingkat Akurasi DT: 98.17%

              precision    recall  f1-score   support

      Rendah       0.99      1.00      1.00      1153
      Sedang       0.98      0.98      0.98      1394
      Tinggi       0.95      0.94      0.95       453

    accuracy                           0.98      3000
   macro avg       0.98      0.97      0.98      3000
weighted avg       0.98      0.98      0.98      3000

